# 🧪Lab: Predicting Seismic Building Response with Support Vector Regression

In this lab, you will practice **Support Vector Machine (SVM)** in the context of **regression**. Over the past two weeks, we have focused on SVM for **classification problems**. However, SVM can also be extended to handle **regression tasks**—this is known as **Support Vector Regression (SVR)**.

To explore this, you will work with the following paper:  https://www.nature.com/articles/s41598-024-81705-3

In this paper, the authors use **real seismic monitoring data** to develop an SVR-based model for predicting the **maximum inter-story drift ratio** of buildings during earthquakes. The authors compare a full-feature model with a reduced-feature version, and demonstrate that SVR is effective in this complex, real-world regression setting.

You will begin by reading the **introduction** of the paper to understand the motivation behind the problem. Later, you will apply Support Vector Regression to their dataset.

Reference:

- Tao, D., Fang, S., Liu, H. et al. Support vector regression model for the prediction of buildings’ maximum seismic response based on real monitoring data. Sci Rep 14, 29874 (2024). https://doi.org/10.1038/s41598-024-81705-3

--- 

**Software**: Use `scikit-learn` and its implementation of Support Vector Regression in this lab. https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

--- 

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


NAME(s) = "Namitha Tholasi, Chloe Wang, Ethan Ooi"

## 1. Paper reflection (10 points)

a. After reading the introduction of the paper, discuss within the group and address the following questions:
   
   - Why is predicting seismic building response important? (You may consider implications for human safety, economic losses, emergency planning, etc)

- Why might the dataset of this study be particularly well-suited for a machine learning approach? (You may think about the volume and diversity of data, the types of features provided, and the limitations of traditional physics-based methods).

Please, elaborate on your answers.

**Why is predicting seismic building response important?**

Earthquakes can seriously damage or collapse buildings, which leads to injuries, deaths, and huge financial losses (about $14.7 billion a year in the U.S. just from building damage). After an earthquake, a city might have hundreds of thousands of buildings that need to be checked, and there isn't time to inspect them all right away. If we can quickly predict how much each building moved or deformed, we can estimate which ones are most likely damaged. That helps emergency teams decide where to go first, figure out which buildings are unsafe to enter, and estimate the overall cost of the damage for recovery planning.

**Why is this dataset well-suited for machine learning?**

Most earlier studies trained models on computer simulations, but this dataset comes from sensors in real buildings during real earthquakes, so the model learns from how buildings actually behave. It's also pretty large, with thousands of recordings from over 100 buildings and thousands of earthquake events. It has 41 different features covering the building (height, number of stories, natural frequency), the shaking (spectral values, duration), and the earthquake itself (magnitude, distance, peak ground motion). The relationship between all these features and building drift is complicated and nonlinear, which is hard for simple linear regression but a good fit for ML. Traditional physics-based models are accurate but slow, and they need detailed design information that most buildings don't have. An ML model can make predictions almost instantly using basic information, which is what you need right after an earthquake.

b. After reading the paper, explain, in your own words, how Support Vector Regression (SVR) works. In addition, explain Formulas (1) to (11) from the paper, specifically, what is the intuition behind each equation in light of what we covered in class regarding Support Vector Machine, so you can see the correspondence between SVM for classification and regression.

**SVR is SVM adapted for regression. Instead of separating classes, SVR fits a line or curve and puts a "tube" of width ε around it. Points inside the tube count as no error, and only points outside the tube get penalized. Like SVM, it keeps the model simple by minimizing $\|w\|^2$, uses C to control how strict it is about errors, relies only on a few key points called support vectors, and can use kernels to fit curved patterns.**

Formulas (1) to (11)

(1) $f(x) = w^T\phi(x) + b$
The prediction function. Same form as the SVM hyperplane, but the output is the predicted value.

(2) $\min \; \frac{1}{2}\|w\|^2 + C\sum L_\varepsilon$
The goal is to keep the model simple while keeping errors small, just like soft-margin SVM.

(3) ε-insensitive loss
Errors smaller than ε count as zero. This is SVR's version of the hinge loss.

(4) $\min \; \frac{1}{2}\|w\|^2 + C\sum(\xi_i + \xi_i^*)$
Same goal using slack variables, one for points above the tube and one for points below.

(5) Constraints
Every point has to be inside the tube unless it uses slack. SVM has one constraint per point, while SVR has two (one for each side).

(6) Lagrangian
Combines the goal and the constraints into one equation, like setting up the SVM dual problem in class.

(7) $w = \sum(\alpha_i - \alpha_i^*)\phi(x_i)$
$w$ is a weighted sum of the training points, like in SVM.

(8) $\sum(\alpha_i - \alpha_i^*) = 0$
A balance condition, like $\sum \alpha_i y_i = 0$ in SVM.

(9) $0 \le \alpha_i, \alpha_i^* \le C$
Points inside the tube get $\alpha = 0$ and don't affect the model. Only the support vectors matter.

(10) $f(x) = \sum(\alpha_i - \alpha_i^*)K(x_i, x) + b$
The final model written with a kernel. The kernel trick lets SVR fit nonlinear patterns. The paper uses the RBF kernel.

(11) Finding $b$
We solve for $b$ using a support vector on the edge of the tube, then use (10) to make predictions. SVM finds $b$ the same way.

## 2. Exploratory Analysis (15 Points)

a. Load the dataset accompanying this lab and create a new dataset containing the same input features used as predictors in the paper, along with the target variable, *Drift*. **You will use only this new dataset throughout the remainder of the lab.** Display the first few rows of the resulting dataset.

*Hint*: The exact input features used in the predictive framework are clearly specified in the paper. You may also refer to Table 3 to help identify the corresponding features in the provided dataset.

In [1]:
# Build X and y (41 input features from Table 3 of the paper + Drift)
import pandas as pd

df = pd.read_excel('/Users/chloewang/Desktop/DS_4021/DS_4021_Labs/02_Lab/data_lab02 (1).xlsx', header=1, skiprows=[2])

# Paper names in comments, dataset column names in the list
feature_cols = [
    "B_Lat.", "B_Long.", "Height", "No. of story", "F1", "F2",
    "SA1", "SV1", "SD1", "SA2", "SV2", "SD2", "Avg_Sa", "Avg_Sv", "Avg_Sd",
    "E_Lat.", "E_Long.", "Magnitude", "Epicentral distance (R)",
    "PGA", "PGV", "PGD", "AI", "CAV", "DP",
    "ZX", "Effective",
    "Bracketed 1 [0.05g]", "Bracketed [0.1g]", "Bracketed [0.15g]", "Bracketed [0.2g]",
    "Uniform [0.05g]", "Uniform [0.1g]", "Uniform [0.15g]", "Uniform [0.2g]",
    "Significant_a1 [(5-75)%]", "Significant_a2 [(5-95)%]",
    "Significant_v1 [(5-75)%]", "Significant_v2 [(5-95)%]",
    "Significant_d1 [(5-75)%]", "Significant_d2 [(5-95)%]",
]

lab_df = df[feature_cols + ["Drift"]].dropna()

X = lab_df[feature_cols]
y = lab_df["Drift"]

lab_df.head()

,B_Lat.,B_Long.,Height,No. of story,F1,F2,SA1,SV1,SD1,SA2,...,Uniform [0.1g],Uniform [0.15g],Uniform [0.2g],Significant_a1 [(5-75)%],Significant_a2 [(5-95)%],Significant_v1 [(5-75)%],Significant_v2 [(5-95)%],Significant_d1 [(5-75)%],Significant_d2 [(5-95)%],Drift
0,36.132,140.073,3400.0,8,1.962438,1.596588,2.800003,0.245024,0.018094,2.064495,...,0.0,0.0,0.0,54.89,73.15,56.97,81.65,93.23,122.20,-0.218343
1,36.132,140.073,3400.0,8,1.834042,1.684939,2.581543,0.259164,0.019248,2.205341,...,0.0,0.0,0.0,5.96,19.57,13.88,27.74,25.26,63.18,-0.226958
2,36.132,140.073,3400.0,8,1.846080,1.696926,0.710809,0.083139,0.005131,0.671577,...,0.0,0.0,0.0,10.60,29.11,18.58,48.81,70.65,72.53,-0.240754
3,36.132,140.073,3400.0,8,1.949488,1.580632,0.578160,0.053573,0.003708,0.366055,...,0.0,0.0,0.0,10.45,25.24,13.61,32.37,33.76,57.79,-0.244348
4,36.132,140.073,3400.0,8,1.778042,1.144375,12.618900,0.986632,0.099006,22.317243,...,0.0,0.0,0.0,24.25,48.01,23.06,50.12,28.47,70.65,-0.101461


b. Explore the distribution of the target variable (Drift), and create scatterplots between Drift and input features.

In [2]:
# [Your code here]

c. Highlight any particular pattern you observe from the visualizations. Which input features seem potentially more influential in predicting Drift?

YOUR TEXT HERE

## 3. Fit and evaluate (30 points)

a. Fit and evaluate a predictive framework using an SVR model. The process should include automatic tuning of both the kernel and the penalty parameter ((C)). Tune the hyperparameters using cross-validation on the training set, and evaluate the final predictive framework on a separate reporting set. Evaluate its performance using at least the Coefficient of Determination ($R^2$) and Mean Squared Error (MSE).

*N.B.* Remember that a predictive framework may involve more than just the predictive algorithm itself. Consider whether the characteristics of the predictors require any additional steps before fitting the model.

In [3]:
# A. SVR predictive framework
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

# X and y come from Part 2 (paper's input features and Drift)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# SVR is not scale invariant, so the scaler goes inside the pipeline
svr_regressor = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR()),
])

inner_cv = KFold(n_splits=5, shuffle=True, random_state=1234)
candidate_grid = {
    "svr__kernel": ["linear", "poly", "rbf", "sigmoid"],
    "svr__C": [0.1, 1, 10, 100],
}

svr_search = GridSearchCV(
    estimator=svr_regressor,
    param_grid=candidate_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)
svr_search.fit(X_train, y_train)

svr_preds = svr_search.best_estimator_.predict(X_test)
svr_r2 = r2_score(y_test, svr_preds)
svr_mse = mean_squared_error(y_test, svr_preds)

print(f"Selected kernel: {svr_search.best_params_['svr__kernel']}")
print(f"Selected C: {svr_search.best_params_['svr__C']}")
print(f"Best cross-validated R2 on training set: {svr_search.best_score_:.2f}")
print(f"Final testing R2: {svr_r2:.2f}")
print(f"Final testing MSE: {svr_mse:.2f}")

cv_results = pd.DataFrame(svr_search.cv_results_)
cv_results[["param_svr__kernel", "param_svr__C", "mean_test_score", "std_test_score"]]

Selected kernel: linear
Selected C: 0.1
Best cross-validated R2 on training set: 0.46
Final testing R2: 0.50
Final testing MSE: 0.29


,param_svr__kernel,param_svr__C,mean_test_score,std_test_score
0,linear,0.1,4.619355e-01,1.243476e-01
1,poly,0.1,-3.767096e+01,4.399100e+01
2,rbf,0.1,1.412047e-01,6.667871e-02
3,sigmoid,0.1,-5.784610e+01,2.388246e+01
4,linear,1.0,4.583757e-01,1.260662e-01
5,poly,1.0,-2.090394e+02,3.044321e+02
6,rbf,1.0,2.396979e-01,9.015902e-02
7,sigmoid,1.0,-5.743757e+03,2.350801e+03
8,linear,10.0,4.550080e-01,1.229622e-01
9,poly,10.0,-1.555939e+03,2.022557e+03


b. Repeat the previous process using an Ridge regression predictive framework. Be sure to include automatic tuning of the model's hyperparameters, specifically, `alpha`.

In [ ]:
# B. Ridge predictive framework
from sklearn.linear_model import Ridge

ridge_regressor = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge()),
])

candidate_grid = {"ridge__alpha": np.logspace(-4, 4, 50)}

ridge_search = GridSearchCV(
    estimator=ridge_regressor,
    param_grid=candidate_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)

ridge_preds = ridge_search.best_estimator_.predict(X_test)
ridge_r2 = r2_score(y_test, ridge_preds)
ridge_mse = mean_squared_error(y_test, ridge_preds)

print(f"Selected alpha: {ridge_search.best_params_['ridge__alpha']:.4f}")
print(f"Best cross-validated R2 on training set: {ridge_search.best_score_:.2f}")
print(f"Final testing R2: {ridge_r2:.2f}")
print(f"Final testing MSE: {ridge_mse:.2f}")


Selected alpha: 2222.9965
Best cross-validated R2 on training set: 0.48
Final testing R2: 0.50
Final testing MSE: 0.30


c. Use predicted vs. true value scatterplots to compare the SVR and Ridge predictive frameworks. Embed performance metrics (e.g., R², MSE) in each plot.

Which predictive framework appears to perform better at this task?

In [ ]:
# [Your code here]

## 4. Reduced SVR Model (30 points)

a. Add a feature selection step to the SVR predictive framework that selects the 10 features most highly correlated with Drift. Retrain  this updated framework and evaluate its performance.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler

svr_regressor_fs = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func = f_regression, k = 10)),
    ("svr", SVR()),
])

candidate_grid_fs = {
    "svr__kernel": ["linear", "poly", "rbf", "sigmoid"],
    "svr__C": [0.1, 1, 10, 100],
}

svr_search_fs = GridSearchCV(
    estimator=svr_regressor_fs,
    param_grid=candidate_grid_fs,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)

svr_search_fs.fit(X_train, y_train)

svr_preds_fs = svr_search_fs.best_estimator_.predict(X_test)
svr_r2_fs = r2_score(y_test, svr_preds_fs)
svr_mse_fs = mean_squared_error(y_test, svr_preds_fs)

print(f"Selected kernel: {svr_search_fs.best_params_['svr__kernel']}")
print(f"Selected C: {svr_search_fs.best_params_['svr__C']}")
print(f"Best cross-validated R2 on training set: {svr_search_fs.best_score_:.2f}")
print(f"Final testing R2: {svr_r2_fs:.2f}")
print(f"Final testing MSE: {svr_mse_fs:.2f}")

cv_results_fs = pd.DataFrame(svr_search_fs.cv_results_)
cv_results_fs[["param_svr__kernel", "param_svr__C", "mean_test_score", "std_test_score"]]

# The selected 10 features
selector = svr_search_fs.best_estimator_.named_steps["selector"]
selected_features = X_train.columns[selector.get_support()]
print(selected_features.tolist())


Selected kernel: linear
Selected C: 1
Best cross-validated R2 on training set: 0.45
Final testing R2: 0.52
Final testing MSE: 0.29


,param_svr__kernel,param_svr__C,mean_test_score,std_test_score
0,linear,0.1,4.512207e-01,7.593915e-02
1,poly,0.1,-1.360155e+01,1.094441e+01
2,rbf,0.1,2.095118e-01,7.656228e-02
3,sigmoid,0.1,-1.982187e+02,9.733341e+01
4,linear,1.0,4.514812e-01,7.440212e-02
5,poly,1.0,-1.431540e+02,2.488173e+02
6,rbf,1.0,2.715907e-01,7.419650e-02
7,sigmoid,1.0,-1.902991e+04,9.694801e+03
8,linear,10.0,4.505631e-01,7.470373e-02
9,poly,10.0,-7.901125e+01,5.765221e+01


In [8]:
selector = svr_search_fs.best_estimator_.named_steps["selector"]
selected_features = X_train.columns[selector.get_support()]
print(selected_features.tolist())

['SA1', 'SV1', 'SD1', 'SV2', 'Avg_Sv', 'PGA', 'PGV', 'PGD', 'CAV', 'Bracketed [0.2g]']


Subsequently, answer the following questions:

- How much does the performance change? \
**For the regular SVR, the selected kernel is linear, the selected c is 0.1. The best cross-validated R2 on the training set is 0.46, the the final testing R2 is 0.50, and the final testing MSE is 0.29. For the reduced SVR after selecting 10 features, the selected kernel is still linear but the selected c is 1.1. The best cross-validated R2 on the training set is 0.45, the final testing R2 is 0.52, and the final testing MSE is 0.29. We typically want a higher R2 and a lower MSE. From the results, the performance of both models are marginally different. The the cross validated R2 went down by 0.01 in the reduced version. However, the final testing R2 went up by 0.02 and the final testing MSE stayed the same. Performance is basically the same and changed very slightly, which indicates that restricting the model to the 10 most correlated features is comparable to a model with all the features while also reducing the number of input features. This is just like PCA, except the no new features are created. Dimension is reduced by picking the existing features that correlate the strongest.**

- What are the practical advantages of using this predictive framework with a smaller subset of features?\
**For a typical model, less input features means less runtime, lower storage, and higher efficiency. While running for the 10 most correlated features actually extended runtime, the resulting reduced dataset may have less computational costs and higher efficiency with the final model. Especially since the results from the reduced version are basically the same as the one from the full version. This means that majority of the variances can be explained by the 10 features most correlated. There is no need for all the other input features because they would add no value to the model.**

Elaborate on your answers.

b. Compare the performance of this reduced-feature SVR framework with an optimal Lasso regression framework that uses all available features.

- Do both frameworks select the same features? \
**No, the frameworks select different features. Reduced SVR selects ['SA1', 'SV1', 'SD1', 'SV2', 'Avg_Sv', 'PGA', 'PGV', 'PGD', 'CAV', 'Bracketed [0.2g]'] while the lasso regression framework selects/ends up with ['B_Long.', 'SV1', 'SA2', 'SD2', 'Avg_Sd', 'PGD', 'AI', 'DP'].**
- How do their performances compare?\
**The reduced SVR has a cross-validated R2 of 0.45, a final testing R2 of 0.52, and a final testing MSE of 0.29. The lasso regression model has a cross-validated R2 of 0.49, a final testing R2 of 0.5, and a final testing MSE of 0.3. While the lasso model has a higher cross-validated R2 by about 0.04, it has lower final testing R2 and higher final testing MSE. The performance of both models are again very similar with small differences in testing metrics.**


In [13]:
# [Your code here]
from sklearn.linear_model import Lasso

lasso_regressor = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=100000)),
])

candidate_grid_lasso = {
    "lasso__alpha": np.logspace(-4, 4, 50)
}

lasso_search = GridSearchCV(
    estimator=lasso_regressor,
    param_grid=candidate_grid_lasso,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)

lasso_preds = lasso_search.best_estimator_.predict(X_test)
lasso_r2 = r2_score(y_test, lasso_preds)
lasso_mse = mean_squared_error(y_test, lasso_preds)

print(f"Selected alpha: {lasso_search.best_params_['lasso__alpha']:.4f}")
print(f"Best cross-validated R2 on training set: {lasso_search.best_score_:.2f}")
print(f"Final testing R2: {lasso_r2:.2f}")
print(f"Final testing MSE: {lasso_mse:.2f}")

# Comparison
selector = svr_search_fs.best_estimator_.named_steps["selector"]
selected_features = X_train.columns[selector.get_support()]
print(selected_features.tolist())

lasso_selector = lasso_search.best_estimator_.named_steps["lasso"]
lasso_features = X_train.columns[~np.isclose(lasso_selector.coef_, 0)]
print(lasso_features.tolist())

Selected alpha: 0.0281
Best cross-validated R2 on training set: 0.49
Final testing R2: 0.50
Final testing MSE: 0.30
['SA1', 'SV1', 'SD1', 'SV2', 'Avg_Sv', 'PGA', 'PGV', 'PGD', 'CAV', 'Bracketed [0.2g]']
['B_Long.', 'SV1', 'SA2', 'SD2', 'Avg_Sd', 'PGD', 'AI', 'DP']


## 5: Robustness to Point Removal (10 points)

a. Compare the sensitivity of SVR and Ridge Regression predictive frameworks to small changes in the training set. To achieve this, randomly remove 10 training points, retrain both predictive frameworks, and assess how much the predictions change.

- Which predictive frameworks appears more stable?
- Why?

In [ ]:
# [Your code here]

## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE